In [ ]:
import numpy as np
import cv2

rng = np.default_rng(42) if hasattr(np, 'default_rng') else np.random.default_rng(42)

def add_salt_pepper(img, amount):
    noisy = img.copy()
    mask = rng.random(img.shape)
    noisy[mask < amount / 2] = 0
    noisy[mask > 1 - amount / 2] = 255
    return noisy

def add_gaussian_noise(img, sigma):
    noisy = img.astype(np.float64) + rng.normal(0, sigma, img.shape)
    return np.clip(noisy, 0, 255).astype(np.uint8)

def mse(a, b):
    return np.mean((a.astype(np.float64) - b.astype(np.float64)) ** 2)

In [ ]:
def median_filter_3x3(img):
    padded = np.pad(img, 1, mode="edge")
    h, w = img.shape
    out = np.zeros_like(img)
    for i in range(h):
        for j in range(w):
            out[i, j] = np.median(padded[i:i+3, j:j+3])
    return out

In [ ]:
x = np.linspace(0, 1, 256)
img_demo4 = (np.outer(x, np.ones(256)) * 100 + np.outer(np.ones(256), x) * 100 + 30).astype(np.uint8)

noisy = add_salt_pepper(img_demo4, 0.05)
error = np.max(np.abs(median_filter_3x3(noisy).astype(int) - cv2.medianBlur(noisy, 3).astype(int)))
print(f"median_filter_3x3 vs cv2.medianBlur: max error = {error}")

In [ ]:
from IPython.display import display, Image

noisy_sp = add_salt_pepper(img_demo4, 0.10)
noisy_gs = add_gaussian_noise(img_demo4, 20)

med_sp = median_filter_3x3(noisy_sp)
gau_sp = cv2.GaussianBlur(noisy_sp, (3, 3), 0)

med_gs = median_filter_3x3(noisy_gs)
gau_gs = cv2.GaussianBlur(noisy_gs, (3, 3), 0)

mse_med_sp = mse(img_demo4, med_sp)
mse_gau_sp = mse(img_demo4, gau_sp)
mse_med_gs = mse(img_demo4, med_gs)
mse_gau_gs = mse(img_demo4, gau_gs)

def make_cell(img_gray, title_text):
    cell = cv2.resize(img_gray, (220, 220))
    bgr = cv2.cvtColor(cell, cv2.COLOR_GRAY2BGR)
    canvas = np.ones((260, 220, 3), dtype=np.uint8) * 255
    canvas[40:, 0:220] = bgr
    cv2.putText(canvas, title_text, (10, 26), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 0), 1, cv2.LINE_AA)
    return canvas

c1 = make_cell(img_demo4, "Original")
c2 = make_cell(noisy_sp, "Salt & Pepper 10%")
c3 = make_cell(med_sp, f"Median MSE={mse_med_sp:.1f}")
c4 = make_cell(gau_sp, f"Gauss MSE={mse_gau_sp:.1f}")
row1 = np.hstack([c1, c2, c3, c4])

c5 = make_cell(img_demo4, "Original")
c6 = make_cell(noisy_gs, "Gaussian sigma=20")
c7 = make_cell(med_gs, f"Median MSE={mse_med_gs:.1f}")
c8 = make_cell(gau_gs, f"Gauss MSE={mse_gau_gs:.1f}")
row2 = np.hstack([c5, c6, c7, c8])

full_grid = np.vstack([row1, row2])
cv2.imwrite("comparison_result.png", full_grid)
print("Đã lưu thành công file comparison_result.png!")
display(Image("comparison_result.png"))

In [ ]:
sp_levels = [0.01, 0.05, 0.10, 0.15, 0.20, 0.30]
sigmas = [5, 10, 20, 30, 40, 50]

sp_med, sp_gauss = [], []
for amount in sp_levels:
    noisy = add_salt_pepper(img_demo4, amount)
    sp_med.append(mse(img_demo4, median_filter_3x3(noisy)))
    sp_gauss.append(mse(img_demo4, cv2.GaussianBlur(noisy, (3, 3), 0)))

gs_med, gs_gauss = [], []
for sigma in sigmas:
    noisy = add_gaussian_noise(img_demo4, sigma)
    gs_med.append(mse(img_demo4, median_filter_3x3(noisy)))
    gs_gauss.append(mse(img_demo4, cv2.GaussianBlur(noisy, (3, 3), 0)))

def draw_chart(title, x_vals, y_med, y_gau, x_label):
    img = np.ones((400, 600, 3), dtype=np.uint8) * 255
    cv2.putText(img, title, (150, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 2, cv2.LINE_AA)
    cv2.putText(img, x_label, (230, 385), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
    cv2.putText(img, "MSE", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
    
    cv2.line(img, (70, 330), (550, 330), (100, 100, 100), 1)
    cv2.line(img, (70, 50), (70, 330), (100, 100, 100), 1)
    
    max_y = max(max(y_med), max(y_gau)) if max(max(y_med), max(y_gau)) > 0 else 1
    
    pts_med, pts_gau = [], []
    for i, xv in enumerate(x_vals):
        px = int(70 + (i / (len(x_vals) - 1)) * 450)
        py_m = int(330 - (y_med[i] / max_y) * 260)
        py_g = int(330 - (y_gau[i] / max_y) * 260)
        pts_med.append((px, py_m))
        pts_gau.append((px, py_g))
        
        cv2.circle(img, (px, py_m), 4, (255, 0, 0), -1)
        cv2.circle(img, (px, py_g), 4, (0, 0, 255), -1)

    for i in range(len(pts_med) - 1):
        cv2.line(img, pts_med[i], pts_med[i+1], (255, 0, 0), 2, cv2.LINE_AA)
        cv2.line(img, pts_gau[i], pts_gau[i+1], (0, 0, 255), 2, cv2.LINE_AA)
        
    cv2.putText(img, "Median 3x3", (430, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 0, 0), 1, cv2.LINE_AA)
    cv2.putText(img, "Gaussian 3x3", (430, 95), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 255), 1, cv2.LINE_AA)
    return img

chart1 = draw_chart("Salt & Pepper Noise", sp_levels, sp_med, sp_gauss, "Density")
chart2 = draw_chart("Gaussian Noise", sigmas, gs_med, gs_gauss, "Sigma")
final_chart = np.hstack([chart1, chart2])

cv2.imwrite("MSE.png", final_chart)
print("Đã tạo và lưu biểu đồ MSE thành công vào file MSE.png!")
display(Image("MSE.png"))

print("Salt & pepper:")
for a, m, g in zip(sp_levels, sp_med, sp_gauss):
    print(f"  {a:.2f}: median={m:7.1f}  gaussian={g:7.1f} -> {'median' if m < g else 'gaussian'} wins")

print("Gaussian noise:")
for s, m, g in zip(sigmas, gs_med, gs_gauss):
    print(f"  σ={s:2d}: median={m:7.1f}  gaussian={g:7.1f} -> {'median' if m < g else 'gaussian'} wins")